In [0]:
%sql
create or replace view vw_fact_sale_sk as
  select
     tss.invoice_id
    ,tss.part_quantity
    ,tss.part_unit_price
    ,tss.part_unit_cost
    ,tss.part_total_price
    ,tss.part_total_cost
    ,tss.part_profit
    ,coalesce(td._sk_date, -1)     as _sk_date
    ,coalesce(tc._sk_client, -1)   as _sk_client
    ,coalesce(tcar._sk_car, -1)    as _sk_car
    ,coalesce(tst._sk_store, -1)   as _sk_store
    ,coalesce(ts._sk_seller, -1)   as _sk_seller
    ,tss.sale_date
    ,tss.ingestion_timestamp
  from      workspace.pj_sales.tb_sales_silver tss
  left join workspace.pj_sales.tb_dim_date_gold td
    on      tss.sale_date = td.date
  left join workspace.pj_sales.tb_dim_seller_gold ts
    on      tss.seller_id = ts.seller_id
    and     (tss.sale_date < ts.valid_to
    or      ts.valid_to is null)
  left join workspace.pj_sales.tb_dim_car_gold tcar
    on      tss.car_brand = tcar.car_brand
    and     tss.car_model = tcar.car_model
    and     tss.car_part = tcar.car_part
    and     (tss.sale_date < tcar.valid_to 
    or      tcar.valid_to is null) -- Trata e pega a versão da dimensão referente a data da venda
  left join workspace.pj_sales.tb_dim_store_gold tst
    on      tss.store_id = tst.store_id
    and     (tss.sale_date < tst.valid_to 
    or      tst.valid_to is null)
  left join workspace.pj_sales.tb_dim_client_gold tc
    on      tss.client_id = tc.client_id
    and     (tss.sale_date < tc.valid_to 
    or      tc.valid_to is null)

In [0]:
%sql
create or replace temp view vw_fact_sale as(
  with dedup as (
    select
       invoice_id
      ,part_quantity
      ,part_unit_price
      ,part_unit_cost
      ,part_total_price
      ,part_total_cost
      ,part_profit
      ,_sk_date
      ,_sk_client
      ,_sk_car
      ,_sk_store
      ,_sk_seller
      ,row_number() over (
        partition by
           invoice_id
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      ) as order_by_date
    from vw_fact_sale_sk
  )
  select
     invoice_id
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,part_total_price
    ,part_total_cost
    ,part_profit
    ,_sk_date
    ,_sk_client
    ,_sk_car
    ,_sk_store
    ,_sk_seller
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
delete from workspace.pj_sales.tb_fact_sale_gold
where invoice_id in (
  select distinct
    invoice_id
  from vw_fact_sale
)

In [0]:
%sql
insert into workspace.pj_sales.tb_fact_sale_gold (
     invoice_id
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,part_total_price
    ,part_total_cost
    ,part_profit
    ,_sk_date
    ,_sk_client
    ,_sk_car
    ,_sk_store
    ,_sk_seller
  )
  select
    *
  from vw_fact_sale